# BODAQS Batch Pre-processing Pipeline

Workflow:
1. Load CSVs
2. Canonicalise signal names
3. Normalize (zero + scale)
4. Estimate velocity/acceleration
5. Load event schema + detect events
6. Calculate event metrics
7. Write artifacts


In [1]:
import pandas as pd
import numpy as np
import logging
import importlib
import json
import os

from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df
from bodaqs_analysis.pipeline import load_and_canonicalize
import bodaqs_analysis.widgets.event_browser as eb
importlib.reload(eb)

from bodaqs_analysis.artifacts import (
    ArtifactStore,
    make_run_id,
    save_session_artifacts,
    write_run_manifest,
    write_session_manifest,
    write_events_partitioned_by_schema_id,
    write_metrics_partitioned_by_schema_id,
    copy_raw_csv_to_source,
    ensure_run_is_new,
    ensure_session_is_new,
    update_manifest_description,
)

root = logging.getLogger()
root.setLevel(logging.WARNING)

# Ensure any existing handlers (added by Jupyter) also filter at WARNING
for h in root.handlers:
    h.setLevel(logging.WARNING)

# Optional: ensure BODAQS loggers don’t override root
logging.getLogger("bodaqs_analysis").setLevel(logging.NOTSET)  # inherit

fmt = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
root = logging.getLogger()
root.setLevel(logging.WARNING)

for h in root.handlers:
    h.setLevel(logging.WARNING)
    h.setFormatter(fmt)


In [2]:
from IPython.display import display
from bodaqs_analysis.ui.preprocess_file_selector import PreprocessLogSelector

selector = PreprocessLogSelector(artifacts_dir=Path("artifacts"))
display(selector.ui)



In [3]:
CSV_FILES = selector.get_selected_files()
if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logging.info("Found %d files:", len(CSV_FILES))
for p in CSV_FILES:
    logging.info("  %s", p.name)

# ---- Step 1: Load + canonicalise (no preprocess params yet) ----
sessions_step1 = {}
disp_cols_union = set()

for p in CSV_FILES:
    s = load_and_canonicalize(str(p))
    sid = str(s["session_id"])
    sessions_step1[sid] = s

    sig = (s.get("meta", {}) or {}).get("signals", {}) or {}
    disp_cols = [c for c, info in sig.items() if isinstance(info, dict) and info.get("quantity") == "disp"]
    disp_cols_union.update(disp_cols)

disp_cols_all = sorted(disp_cols_union)

logging.info("Displacement signals detected:")
for c in disp_cols_all:
    logging.info("  %s", c)

# (Optional) show any unclassified numeric signals so you can fix naming upstream
unclassified = []
for sid, s in sessions_step1.items():
    sig = (s.get("meta", {}) or {}).get("signals", {}) or {}
    for c, info in sig.items():
        if isinstance(info, dict) and info.get("quantity") is None:
            unclassified.append((sid, c, info.get("notes")))
if unclassified:
    print("\nUnclassified numeric columns (may need naming/normalization attention):")
    for sid, c, note in unclassified[:50]:
        print(f"  [{sid}] {c}  ({note})")

In [4]:
from bodaqs_analysis.pipeline import run_macro
from bodaqs_analysis.artifacts import (
    ArtifactStore, make_run_id, ensure_run_is_new, ensure_session_is_new,
    copy_raw_csv_to_source, save_session_artifacts,
    write_session_manifest, write_run_manifest,
    write_events_partitioned_by_schema_id, write_metrics_partitioned_by_schema_id,
)

from bodaqs_analysis.ui.preprocess_controls import PreprocessControls, PreprocessDefaults
import ipywidgets as W
from IPython.display import display

# disp_cols_all and sessions_step1 come from Step 1
controls = PreprocessControls(
    disp_cols_all=disp_cols_all,
    sessions_by_id=sessions_step1,
    defaults=PreprocessDefaults(
        schema_path=r"event schema\event_schema.yaml",
        ingestion_mode="tolerant",
        prompt_for_descriptions=True,
    ),
    default_ranges={
        # optional: prefill some known defaults
        "front_shock_dom_suspension [mm]": 170.0,
        "rear_shock_dom_suspension [mm]": 150.0,
    }
)

display(controls.ui)



Accordion(children=(VBox(children=(Text(value='event schema\\event_schema.yaml', description='Schema path', la…

In [8]:
cfg = controls.get_config()

# If you want hard-stop validation:
errors, warnings = controls.validate(print_to_output=False)
if errors:
    raise ValueError("Fix UI errors before running:\n - " + "\n - ".join(errors))

SCHEMA_PATH = cfg["schema_path"]
PROMPT_FOR_DESCRIPTIONS = cfg["prompt_for_descriptions"]

# ---- Step 2: Validate ranges (from UI config) ----
NORMALIZE_RANGES = cfg["normalize_ranges"]
ZEROING_ENABLED = cfg["zeroing_enabled"]

missing_ranges = [c for c in disp_cols_all if c not in NORMALIZE_RANGES]
if missing_ranges:
    raise ValueError(
        "normalize_ranges is missing displacement signals:\n  - " + "\n  - ".join(missing_ranges)
    )

# ---- Artifact init (one run_id per batch) ----
store = ArtifactStore(Path("artifacts"))
run_id = make_run_id(tz_label="AWST")
ensure_run_is_new(store, run_id=run_id, force=False)
logging.info("Artifact run_id = %s", run_id)

events_all = []
metrics_all = []
sessions_by_id = {}
schema = None
session_ids_in_run = []

for p in CSV_FILES:
    logging.info("Processing %s ...", p.name)

    results = run_macro(
        str(p),
        SCHEMA_PATH,
        zeroing_enabled=cfg["zeroing_enabled"],
        zero_window_s=cfg["zero_window_s"],
        zero_min_samples=cfg["zero_min_samples"],
        clip_0_1=cfg["clip_0_1"],
        active_signal_disp_col=cfg["active_signal_disp_col"],
        active_signal_vel_col=None,
        active_disp_thresh=cfg["active_disp_thresh"],
        active_vel_thresh=cfg["active_vel_thresh"],
        active_window=cfg["active_window"],
        active_padding=cfg["active_padding"],
        active_min_seg=cfg["active_min_seg"],
        normalize_ranges=cfg["normalize_ranges"],
        strict=cfg["strict"],
    )

    session = results["session"]
    sid = str(session["session_id"])

    ensure_session_is_new(store, run_id=run_id, session_id=sid, force=False)
    sessions_by_id[sid] = session
    session_ids_in_run.append(sid)

    source_sha256 = copy_raw_csv_to_source(store=store, run_id=run_id, session_id=sid, csv_path=p)

    save_session_artifacts(
        store,
        run_id=run_id,
        session_id=sid,
        session_df=session["df"],
        session_meta=session["meta"],
    )

    session["df"].to.csv(r"df_dump.csv",index=False)
    
    write_session_manifest(
        store,
        run_id=run_id,
        session_id=sid,
        contracts={"session": "v0.x", "events": "v0.x", "metrics": "v0.x"},
        source={"path": "source/input.csv", "sha256": source_sha256},
        summary={"n_rows": int(len(session["df"]))},
    )

    if schema is None:
        schema = results.get("schema", None)

    ev = results.get("events")
    mt = results.get("metrics")

    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
        write_events_partitioned_by_schema_id(
            store=store,
            run_id=run_id,
            session_id=sid,
            events_df=ev,
            schema_path=SCHEMA_PATH,
        )

    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)
        write_metrics_partitioned_by_schema_id(
            store=store,
            run_id=run_id,
            session_id=sid,
            metrics_df=mt,
        )

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

write_run_manifest(
    store,
    run_id=run_id,
    session_ids=session_ids_in_run,
    timezone_label="AWST",
    pipeline_config={
        "schema_path": str(SCHEMA_PATH),
        "zeroing_enabled": bool(cfg["zeroing_enabled"]),
        "normalize_ranges": bool(cfg["normalize_ranges"]),
        "ingestion_mode": "strict" if cfg["strict"] else "tolerant",
        "n_files": int(len(CSV_FILES)),
    },
)

logging.info("Wrote artifacts for run_id=%s with %d sessions", run_id, len(session_ids_in_run))
# ---- Optional: prompt for run/session descriptions ----
if PROMPT_FOR_DESCRIPTIONS:
    from bodaqs_analysis.artifacts import set_run_description, set_session_description

    run_desc = input(f"Run description for {run_id} (blank to skip): ").strip()
    set_run_description(store, run_id=run_id, description=run_desc)

    for sid in session_ids_in_run:
        s_desc = input(f"Session {sid} description (blank to skip, '.' to stop): ").strip()
        if s_desc == ".":
            break
        set_session_description(store, run_id=run_id, session_id=sid, description=s_desc)

AttributeError: 'DataFrame' object has no attribute 'to'